# Módulo B.2 — Entrenamiento y coparación de modelos

Entrena modelos de Random Forest, XGBoost, SVM y LightGBM, y luego compáralos utilizando dos enfoques: un holdout estratificado y la validación cruzada espacial.

## 1. Instalar librerías

In [1]:
!pip install xgboost lightgbm -q

## 2. Montar Drive y cargar el dataset enriquecido

In [2]:
import os
from pathlib import Path
import pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

CARPETA_BASE = Path('/content/drive/MyDrive/TFM_TeaSuitability')
RUTA_CSV = CARPETA_BASE / 'dataset_master_enriquecido.csv'

if not RUTA_CSV.exists():
    raise FileNotFoundError('Falta el enriquecido. Ejecuta el A.2 real.')

dataset = pd.read_csv(RUTA_CSV)
print('Cargado. Forma:', dataset.shape)

Mounted at /content/drive
Cargado. Forma: (1333, 14)


## 3. Importaciones

In [3]:
import warnings                                    # controlar (aquí, silenciar) los mensajes de aviso
import numpy as np                                 # cálculo numérico con arrays (operaciones rápidas con números)
from lightgbm import LGBMClassifier                # modelo LightGBM (boosting rápido y eficiente)
from sklearn.cluster import KMeans                 # agrupar puntos; aquí, crear los bloques espaciales por coordenadas
from sklearn.ensemble import RandomForestClassifier  # modelo Random Forest
from sklearn.metrics import (accuracy_score,       # medir el % de aciertos (exactitud)
                             f1_score,             # equilibrio entre precisión y recall
                             precision_score,      # medir cuántos de los predichos "té" lo eran de verdad
                             recall_score,         # medir cuántos "té" reales encontró el modelo
                             roc_auc_score)        # medir cómo de bien ordena positivos y negativos (0.5=azar, 1=perfecto)

from sklearn.model_selection import (GroupKFold,   # validación cruzada espacial (mantiene juntos los bloques)
                                     cross_val_score,  # calcular una métrica promediada por validación cruzada
                                     train_test_split)  # dividir los datos en entrenamiento y prueba
from sklearn.pipeline import make_pipeline         # encadenar pasos (aquí, estandarizar + SVM) como un solo modelo
from sklearn.preprocessing import StandardScaler   # estandarizar variables (media 0, desviación 1)
from sklearn.svm import SVC                         # modelo SVM
from xgboost import XGBClassifier                   # modelo XGBoost
warnings.filterwarnings('ignore')                  # oculta los avisos para que la salida quede más limpia

## 4. Preparación (resumen de B.1)

In [4]:
# Predictoras = las 8 variables ambientales (mismo orden que en A.2 y B.1)
columnas_predictoras = [
    'temperatura_media',        # bio1: temperatura media anual (óptimo del té: 18-25 °C)
    'rango_diurno',             # bio2: diferencia entre la máxima del día y la mínima nocturna
    'precipitacion_anual',      # bio12: lluvia total anual (óptimo 1500-3000 mm)
    'estacionalidad_precip',    # bio15: variación de la lluvia entre estaciones
    'precip_trimestre_seco',    # bio17: lluvia en la estación seca (estrés hídrico)
    'elevacion',                # altitud del terreno (el té se cultiva hasta ~2200 m)
    'ph_suelo',                 # pH del suelo (SoilGrids); el té prefiere ácido, 4.5-5.5
]
X = dataset[columnas_predictoras]
y = dataset['clase']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)
# proporción negativos/positivos para que XGBoost atienda a la minoría
proporcion = int((y_train == 0).sum()) / int((y_train == 1).sum())
# bloques espaciales para la validación honesta
bloques = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(dataset[['lon','lat']])

## 5. Funciones de modelado y evaluación

In [5]:
def definir_modelos(proporcion):
    """
    Crea el diccionario de modelos a comparar.

    Parameters
    ----------
    proporcion : float
        Cociente negativos/positivos, para ponderar la clase minoritaria

    Returns
    -------
    dict of {str: estimator}
        Nombre del modelo -> modelo de scikit-learn listo para entrenar
    """
    return {
        'Random Forest': RandomForestClassifier(
            n_estimators=300, class_weight='balanced', random_state=42),
        'XGBoost': XGBClassifier(
            n_estimators=300, scale_pos_weight=proporcion,
            eval_metric='logloss', random_state=42),
        'SVM': make_pipeline(  # SVM necesita estandarización previa
            StandardScaler(),
            SVC(probability=True, class_weight='balanced', random_state=42)),
        'LightGBM': LGBMClassifier(
            n_estimators=300, class_weight='balanced',
            random_state=42, verbose=-1),
    }


def evaluar_modelo(modelo, X_train, y_train, X_test, y_test, X, y, bloques):
    """
    Entrena un modelo y lo evalúa en holdout y en validación espacial

    Parameters
    ----------
    modelo : estimator
        Modelo de scikit-learn sin entrenar.
    X_train, y_train : datos de entrenamiento (holdout).
    X_test, y_test : datos de prueba (holdout).
    X, y : conjunto completo, para la validación cruzada espacial.
    bloques : numpy.ndarray
        Bloque espacial de cada punto.

    Returns
    -------
    dict
        Métricas de holdout (Accuracy, Precision, Recall, F1, ROC-AUC) y
        de validación espacial (F1_espacial, AUC_espacial).
    """
    modelo.fit(X_train, y_train)                       # entrena en el holdout
    pred = modelo.predict(X_test)                      # clases predichas
    proba = modelo.predict_proba(X_test)[:, 1]         # probabilidad de té

    # validación espacial: F1 y ROC-AUC promediados por bloques geográficos
    gkf = GroupKFold(n_splits=len(set(bloques)))
    f1_esp = cross_val_score(modelo, X, y, cv=gkf, groups=bloques,
                             scoring='f1').mean()
    auc_esp = cross_val_score(modelo, X, y, cv=gkf, groups=bloques,
                              scoring='roc_auc').mean()
    return {
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'F1_espacial': f1_esp,
        'AUC_espacial': auc_esp,
    }


def comparar_modelos(modelos, X_train, y_train, X_test, y_test, X, y, bloques):
    """
    Evalúa todos los modelos y devuelve una tabla comparativa

    Parameters
    ----------
    modelos : dict of {str: estimator}
        Modelos a comparar.
    X_train, y_train, X_test, y_test, X, y, bloques
        Datos y bloques (ver evaluar_modelo)

    Returns
    -------
    pandas.DataFrame
        Tabla con una fila por modelo, ordenada por F1 espacial
    """
    filas = []
    for nombre, modelo in modelos.items():
        print('Evaluando', nombre, '...')
        metricas = evaluar_modelo(modelo, X_train, y_train, X_test, y_test,
                                  X, y, bloques)
        filas.append({'Modelo': nombre, **metricas})
    tabla = pd.DataFrame(filas).round(3)
    return tabla.sort_values('F1_espacial', ascending=False).reset_index(drop=True)

## 6. Ejecución: entrenar y comparar

In [6]:
modelos = definir_modelos(proporcion)
tabla = comparar_modelos(modelos, X_train, y_train, X_test, y_test, X, y, bloques)
print()
print(tabla.to_string(index=False))

Evaluando Random Forest ...
Evaluando XGBoost ...
Evaluando SVM ...
Evaluando LightGBM ...

       Modelo  Accuracy  Precision  Recall    F1  ROC-AUC  F1_espacial  AUC_espacial
          SVM     0.928      0.948   0.927 0.937    0.987        0.807         0.953
Random Forest     0.955      0.966   0.957 0.961    0.989        0.773         0.951
     LightGBM     0.950      0.961   0.953 0.957    0.981        0.758         0.946
      XGBoost     0.950      0.969   0.944 0.957    0.981        0.733         0.925


## 7. El mejor modelo

Se opta por **F1 espacial**, que evalúa la habilidad de generalizar en áreas nuevas, siendo una opción más honesta que el método de holdout.

In [7]:
mejor = tabla.iloc[0]['Modelo']
print('Mejor modelo por generalización espacial:', mejor)
print('\nLectura: el holdout da métricas altas (~0.95 F1) pero optimistas por\n'
      'autocorrelación espacial; la validación espacial (F1 ~0.64-0.78) refleja\n'
      'el rendimiento real en territorios no vistos.')

Mejor modelo por generalización espacial: SVM

Lectura: el holdout da métricas altas (~0.95 F1) pero optimistas por
autocorrelación espacial; la validación espacial (F1 ~0.64-0.78) refleja
el rendimiento real en territorios no vistos.
